# Zolai corpus cleaner (v2)

**Notebook:** `zolai-cleaner-v2.ipynb`

Prepare raw Zolai JSONL for training. Output matches **`zolai-qwen-training-v2.ipynb`**: required **`text`** column; optional metadata; optional Hugging Face **`save_to_disk`** export (`dataset_dict.json`) for Hub.

## Run order (top → bottom)

| Step | What |
|------|------|
| 1 | Install |
| 2 | Config + paths + **`log_step`** / **`pipeline.log`** |
| 3 | **Standard clean** (NFKC + whitespace; optional HTML/URL strip) + optional **`bloat_report.json`** (`TRACK_BLOAT_STATS`) |
| 4 | Peek samples |
| 5 | Stats + histogram |
| 6 | Optional soft refine (English UI phrases) |
| 7 | Optional drop rows with functional English noise |
| 8 | Resolve split source |
| 9 | Train / val / test split |
| 10 | Health audit |
| 11 | Optional: `DatasetDict` → disk |
| 12 | Optional: Kaggle dataset new version |
| 13 | Optional: chunk huge JSONL |
| Appendix | Optional custom BPE (from-scratch LM only) |

## Kaggle checklist

- **Internet:** on for `pip` / Hub.
- **Add data:** e.g. **peterpausianlian/zolai-master-data** under `/kaggle/input/datasets/.../`
- **Single file override:** env **`ZOLOAI_INPUT_JSONL`** = path to one `.jsonl`
- **Default:** one file by priority (`zolai_corpus_master.jsonl` before `zolai_train_ready_v2.jsonl`). Set **`MERGE_ALL_INPUT_JSONL = True`** only if you intend to merge multiple sources.

## Standard Zolai text (same idea as training `_clean_one`)

`unicodedata.normalize("NFKC", text)` plus whitespace collapse. Optional HTML/URL removal for noisy sources. **Never** use ASCII-only stripping of “non-letters” — that destroys Zolai text.

## Why the cleaned file can be much smaller than raw (e.g. 9 GB → ~500 MB)

- **Bytes per line, not only row count:** Raw `zolai_corpus_master.jsonl` lines are often **large JSON objects** (`text` full of **HTML**, **URLs**, extra keys). This notebook **strips HTML/URLs** (config: `STRIP_HTML_TAGS`, `STRIP_URLS`), **normalizes** text, and **writes a compact subset** of fields. That cuts **disk size per line** sharply (often **~10–20×**) even when **saved row count ≈ scanned rows**.
- **Deduplication:** With `DEDUPE_EXACT = True`, **identical cleaned lines** are kept once. Check the final **`dupes`** number in the clean step — large dupes also shrink the file.
- **Dropped rows:** Lines below `MIN_TEXT_CHARS` or empty after cleaning are skipped (`short` / `empty` in the summary).
- **To keep more bulk** (e.g. for debugging): set `STRIP_HTML_TAGS = False` and/or `STRIP_URLS = False` in the config cell, or set `DEDUPE_EXACT = False` if you intentionally want repeated lines for LM training.

## Tracking raw bloat (`TRACK_BLOAT_STATS`)

The clean step can write **`bloat_report.json`** next to the log: **total bytes per JSONL line** vs **sum of raw `text` UTF-8 bytes** (shows extra keys + JSON syntax), plus **characters removed** by HTML and URL steps on lines that pass cleaning. **Removing HTML/URLs is usually good for LM training**: less tokenizer noise, fewer useless tokens, better match to how you want the model to write (plain text). Keep stripping off unless you explicitly need markup in the target distribution.

## Progress + `pipeline.log`

- **Install** pulls **`tqdm`** for optional progress bars.
- **Config** defines **`log_step(step_id, message)`** and **`PIPELINE_LOG`** (`zolai_cleaner_out/pipeline.log`). Set **`CLEAR_PIPELINE_LOG_ON_START = False`** to append across re-runs instead of truncating.
- Long loops use **`tqdm`** when **`USE_TQDM = True`**; set **`False`** if bars break in your environment.
- **`cleaning_report.log`** remains the detailed clean pass + optional **`bloat_report`** JSON.


In [5]:
# =========================
# 1. INSTALL
# =========================
# Pins: pandas<3 (gradio etc.); matplotlib<3.10.1 (ydata-profiling); bigquery-storage satisfies Colab preinstalled bigframes.
!pip install -q -U "datasets>=2.19.0" "pandas>=2.0,<3" "scikit-learn" "matplotlib>=3.5,<3.10.1" "google-cloud-bigquery-storage>=2.30.0,<3" "tqdm>=4.66"

import sys
print("[1/13] install OK | Python:", sys.version.split()[0])


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.4/304.4 kB 8.0 MB/s eta 0:00:00:00:01
Python: 3.12.12


In [6]:
# =========================
# 2. CONFIG + PATHS + PIPELINE LOG
# =========================
import os
import re
from datetime import datetime
from pathlib import Path

IS_KAGGLE = os.path.isdir("/kaggle")
WORK_DIR = Path("/kaggle/working" if IS_KAGGLE else os.environ.get("ZOLOAI_WORK_DIR", ".")).resolve()
OUTPUT_DIR = WORK_DIR / "zolai_cleaner_out"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

KAGGLE_MASTER = Path("/kaggle/input/datasets/peterpausianlian/zolai-master-data")
KAGGLE_MASTER_ALT = Path("/kaggle/input/zolai-master-data")

# Priority order (first hit wins unless MERGE_ALL_INPUT_JSONL=True)
INPUT_PRIORITY = [
    "zolai_master_data/zolai_corpus_master.jsonl",
    "zolai_corpus_master.jsonl",
    "zolai_train_ready_v2.jsonl",
    "zolai_master_data/zolai_train_ready_v2.jsonl",
]

MERGE_ALL_INPUT_JSONL = False  # True = concatenate every existing file below (can duplicate corpus)


def _resolve_input_paths():
    env_one = os.environ.get("ZOLOAI_INPUT_JSONL")
    if env_one and Path(env_one).is_file():
        return [Path(env_one)]
    for base in (KAGGLE_MASTER, KAGGLE_MASTER_ALT):
        if not base.is_dir():
            continue
        found = []
        for rel in INPUT_PRIORITY:
            p = base / rel
            if p.is_file():
                found.append(p)
        if not found:
            continue
        if MERGE_ALL_INPUT_JSONL:
            return found
        return [found[0]]
    raise FileNotFoundError(
        "No input JSONL. Add zolai-master-data on Kaggle or set ZOLOAI_INPUT_JSONL."
    )


INPUT_PATHS = _resolve_input_paths()

PRIMARY_JSONL = OUTPUT_DIR / "zolai_train_ready_v2.jsonl"
SOFT_JSONL = OUTPUT_DIR / "zolai_after_soft_refine.jsonl"
GOLD_JSONL = OUTPUT_DIR / "zolai_gold_train.jsonl"
LOG_FILE = OUTPUT_DIR / "cleaning_report.log"
PIPELINE_LOG = OUTPUT_DIR / "pipeline.log"
CLEAR_PIPELINE_LOG_ON_START = True  # truncate pipeline.log when re-running config


def log_step(step_id: str, message: str) -> None:
    """Print + append to PIPELINE_LOG (run this cell before later steps)."""
    ts = datetime.now().isoformat(timespec="seconds")
    line = f"[{ts}] [{step_id}] {message}\n"
    print(line.strip())
    with open(PIPELINE_LOG, "a", encoding="utf-8") as pl:
        pl.write(line)


SEED = 42
MIN_TEXT_CHARS = 10
NORMALIZE_NFKC = True
STRIP_HTML_TAGS = True
STRIP_URLS = True
DEDUPE_EXACT = True

# Short-text logging (sample a few dropped SHORT rows for review)
LOG_SHORT_TEXT = True
SHORT_LOG_MAX_ROWS = 2000
SHORT_LOG_PATH = OUTPUT_DIR / "short_text_samples.jsonl"

RUN_SOFT_REFINE = True
RUN_NOISE_ROW_DROP = False

VAL_TEST_FRAC = 0.02
TEST_WITHIN_TEMP_FRAC = 0.5

EXPORT_HF_DATASET = False
HF_EXPORT_DIR = WORK_DIR / "zolai_hf_disk_export"

KAGGLE_DATASET_ID = os.environ.get("ZOLOAI_KAGGLE_DATASET_ID", "peterpausianlian/zolai-master-data")
KAGGLE_VERSION_MESSAGE = "Zolai cleaner v2 — standard NFKC + pipeline"

# Raw-size diagnostics (cell 3): JSON line vs `text` field, HTML/URL chars removed, output size
TRACK_BLOAT_STATS = True
BLOAT_REPORT_PATH = OUTPUT_DIR / "bloat_report.json"

# Progress bars for long file passes (cell 3+); set False if tqdm causes issues
USE_TQDM = True
# Heartbeat: always print + append clean_progress.log every N lines (even if tqdm is on). 0 = off.
HEARTBEAT_EVERY = 250_000
CLEAN_PROGRESS_LOG = OUTPUT_DIR / "clean_progress.log"

if CLEAR_PIPELINE_LOG_ON_START:
    PIPELINE_LOG.write_text("")

log_step("2-config", "paths resolved")
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Inputs:")
for p in INPUT_PATHS:
    print(" ", p)
print("PRIMARY_JSONL:", PRIMARY_JSONL)

if IS_KAGGLE:
    print("--- sample of /kaggle/input files (max 50) ---")
    _n = 0
    for _root, _, _files in os.walk("/kaggle/input"):
        for _fn in _files:
            if _n < 50:
                print(os.path.join(_root, _fn))
            _n += 1
    print("... total files seen:", _n)


OUTPUT_DIR: /kaggle/working/zolai_cleaner_out
Inputs:
  /kaggle/input/datasets/peterpausianlian/zolai-master-data/zolai_master_data/zolai_corpus_master.jsonl
PRIMARY_JSONL: /kaggle/working/zolai_cleaner_out/zolai_train_ready_v2.jsonl
--- sample of /kaggle/input files (max 50) ---
/kaggle/input/datasets/peterpausianlian/zolai-master-data/zolai_train_ready_v2.jsonl
/kaggle/input/datasets/peterpausianlian/zolai-master-data/zolai_master_data/zolai_cleaned_sentences_hot.jsonl
/kaggle/input/datasets/peterpausianlian/zolai-master-data/zolai_master_data/refined_master_zolai.jsonl
/kaggle/input/datasets/peterpausianlian/zolai-master-data/zolai_master_data/zolai_unified_hot.jsonl
/kaggle/input/datasets/peterpausianlian/zolai-master-data/zolai_master_data/zolai_english_parallel.jsonl
/kaggle/input/datasets/peterpausianlian/zolai-master-data/zolai_master_data/zolai_corpus_master.jsonl
/kaggle/input/datasets/peterpausianlian/zolai-master-data/zolai_master_data/stream_cleaning.log
/kaggle/input/data

In [ ]:
# =========================
# 3. STANDARD ZOLAI CLEAN (streaming)
# =========================
import hashlib
import json
import sys
import time
import unicodedata
from datetime import datetime

RE_WS = re.compile(r"\s+")
RE_HTML = re.compile(r"<[^>]+>")
RE_URL = re.compile(r"https?://\S+|www\.\S+", re.I)


def clean_zolai_text(raw: str):
    """Returns (cleaned_or_None, status, metrics). metrics used when TRACK_BLOAT_STATS."""
    m = {"raw_chars": 0, "html_removed": 0, "url_removed": 0}
    if raw is None:
        return None, "EMPTY", m
    t = str(raw).strip()
    m["raw_chars"] = len(t)
    if not t:
        return None, "EMPTY", m
    if NORMALIZE_NFKC:
        t = unicodedata.normalize("NFKC", t)
    t = RE_WS.sub(" ", t).strip()
    if STRIP_HTML_TAGS:
        before = len(t)
        t = RE_HTML.sub(" ", t)
        t = RE_WS.sub(" ", t).strip()
        m["html_removed"] = max(0, before - len(t))
    if STRIP_URLS:
        before = len(t)
        t = RE_URL.sub(" ", t)
        t = RE_WS.sub(" ", t).strip()
        m["url_removed"] = max(0, before - len(t))
    if len(t) < MIN_TEXT_CHARS:
        return None, "SHORT", m
    return t, "KEEP", m


def text_hash(s: str) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()


def _fmt_bytes(n: int) -> str:
    for u, s in ((1 << 30, "GiB"), (1 << 20, "MiB"), (1 << 10, "KiB")):
        if n >= u:
            return f"{n / u:.2f} {s}"
    return f"{n} B"


def process_all_datasets():
    seen = set()
    stats = {
        "total": 0,
        "saved": 0,
        "errors": 0,
        "short": 0,
        "dupes": 0,
        "empty": 0,
        "json_line_bytes": 0,
        "text_field_bytes_raw": 0,
        "html_removed_chars": 0,
        "url_removed_chars": 0,
        "cleaned_text_bytes_written": 0,
        "raw_text_chars_keep_candidates": 0,
    }
    short_logged = 0
    if LOG_SHORT_TEXT:
        SHORT_LOG_PATH.write_text("", encoding="utf-8")
    t0 = time.time()
    log_step("3-clean", "start standard clean")
    if HEARTBEAT_EVERY:
        CLEAN_PROGRESS_LOG.write_text(
            f"start {datetime.now().isoformat(timespec='seconds')} HEARTBEAT_EVERY={HEARTBEAT_EVERY}\n",
            encoding="utf-8",
        )
        print(
            f"Started: {datetime.now().isoformat(timespec='seconds')} | "
            f"heartbeat every {HEARTBEAT_EVERY:,} lines -> {CLEAN_PROGRESS_LOG} | "
            f"more lines after 2M is normal if the corpus is larger",
            flush=True,
        )
    else:
        print(
            f"Started: {datetime.now().isoformat(timespec='seconds')} | HEARTBEAT_EVERY=0 (no mid-run heartbeats)",
            flush=True,
        )
    with open(PRIMARY_JSONL, "w", encoding="utf-8") as fout, open(LOG_FILE, "w", encoding="utf-8") as flog:
        flog.write(f"inputs={INPUT_PATHS!s}\n")
        for src in INPUT_PATHS:
            print("Processing:", src.name, flush=True)
            with open(src, "r", encoding="utf-8", errors="replace") as fin:
                tqdm_mod = None
                if USE_TQDM:
                    from tqdm.auto import tqdm

                    tqdm_mod = tqdm
                    _it = tqdm(
                        fin,
                        desc=src.name[:48],
                        unit="line",
                        mininterval=1.0,
                        file=sys.stdout,
                        dynamic_ncols=True,
                    )
                else:
                    _it = fin
                for line in _it:
                    stats["total"] += 1
                    stats["json_line_bytes"] += len(line.encode("utf-8"))
                    try:
                        obj = json.loads(line)
                        raw = obj.get("text", "")
                        raw_s = raw if isinstance(raw, str) else str(raw or "")
                        stats["text_field_bytes_raw"] += len(raw_s.encode("utf-8"))
                        cleaned, status, metrics = clean_zolai_text(raw)
                        if status != "KEEP":
                            if status == "SHORT":
                                stats["short"] += 1
                                if LOG_SHORT_TEXT and short_logged < SHORT_LOG_MAX_ROWS:
                                    sample = {
                                        "source": src.name,
                                        "line": stats["total"],
                                        "raw_text": raw_s,
                                        "raw_len": len(raw_s),
                                        "cleaned_text": cleaned,
                                        "cleaned_len": len(cleaned),
                                        "min_text_chars": MIN_TEXT_CHARS,
                                        "cleaned_at": datetime.now().isoformat(timespec="seconds"),
                                    }
                                    with open(SHORT_LOG_PATH, "a", encoding="utf-8") as sl:
                                        sl.write(json.dumps(sample, ensure_ascii=False) + "\n")
                                    short_logged += 1
                            else:
                                stats["empty"] += 1
                            continue
                        if DEDUPE_EXACT:
                            h = text_hash(cleaned)
                            if h in seen:
                                stats["dupes"] += 1
                                continue
                            seen.add(h)
                        stats["html_removed_chars"] += metrics.get("html_removed", 0)
                        stats["url_removed_chars"] += metrics.get("url_removed", 0)
                        stats["raw_text_chars_keep_candidates"] += metrics.get("raw_chars", 0)
                        row = {
                            "text": cleaned,
                            "language": obj.get("language", "Zolai"),
                            "dialect": obj.get("dialect", "Tedim"),
                            "source": src.name,
                            "translation_en": obj.get("translation_en"),
                            "cleaned_at": datetime.now().isoformat(timespec="seconds"),
                        }
                        out_line = json.dumps(row, ensure_ascii=False) + "\n"
                        stats["cleaned_text_bytes_written"] += len(cleaned.encode("utf-8"))
                        fout.write(out_line)
                        stats["saved"] += 1
                    except Exception as e:
                        stats["errors"] += 1
                        flog.write(f"line {stats['total']}: {e}\n")
                    if HEARTBEAT_EVERY and stats["total"] % HEARTBEAT_EVERY == 0:
                        elapsed = time.time() - t0
                        hb = (
                            f"[{datetime.now().isoformat(timespec='seconds')}] "
                            f"scanned={stats['total']:,} saved={stats['saved']:,} "
                            f"dupes={stats['dupes']:,} elapsed_s={elapsed:.0f}"
                        )
                        if tqdm_mod is not None:
                            tqdm_mod.write(hb)
                        else:
                            print(hb, flush=True)
                        with open(CLEAN_PROGRESS_LOG, "a", encoding="utf-8") as cpl:
                            cpl.write(hb + "\n")
                            cpl.flush()
    elapsed = time.time() - t0
    summary = (
        f"Done in {elapsed/60:.2f} min | scanned {stats['total']:,} | saved {stats['saved']:,} | "
        f"dupes {stats['dupes']:,} | short {stats['short']:,} | empty {stats['empty']:,} | errors {stats['errors']:,}"
    )
    print(summary, flush=True)
    if HEARTBEAT_EVERY:
        with open(CLEAN_PROGRESS_LOG, "a", encoding="utf-8") as cpl:
            cpl.write(summary + "\n")
    with open(LOG_FILE, "a", encoding="utf-8") as flog:
        flog.write(summary + "\n")

    if TRACK_BLOAT_STATS:
        jl, tr = stats["json_line_bytes"], stats["text_field_bytes_raw"]
        overhead = max(0, jl - tr)
        report = {
            "input_jsonl_total_bytes": jl,
            "sum_raw_text_field_bytes": tr,
            "approx_non_text_json_overhead_bytes": overhead,
            "html_chars_removed_on_kept_lines": stats["html_removed_chars"],
            "url_chars_removed_on_kept_lines": stats["url_removed_chars"],
            "utf8_bytes_unique_cleaned_text_written": stats["cleaned_text_bytes_written"],
            "rows_scanned": stats["total"],
            "rows_saved": stats["saved"],
            "note": "Overhead = full JSONL line bytes minus raw `text` UTF-8 (other keys + JSON syntax). HTML/URL counts apply to lines that passed clean() before dedupe.",
        }
        if tr > 0:
            report["ratio_json_lines_to_raw_text"] = round(jl / tr, 3)
        if stats["cleaned_text_bytes_written"] > 0 and tr > 0:
            report["ratio_raw_text_bytes_to_clean_text_bytes"] = round(tr / stats["cleaned_text_bytes_written"], 3)
        with open(BLOAT_REPORT_PATH, "w", encoding="utf-8") as bf:
            json.dump(report, bf, indent=2)
        print("\n--- Raw bloat report (why input was large) ---")
        print(f"  Input JSONL (sum of line bytes):     {_fmt_bytes(jl)}")
        print(f"  Sum of raw `text` field bytes:       {_fmt_bytes(tr)}")
        print(f"  Approx. non-text overhead (JSON):    {_fmt_bytes(overhead)}  (other keys + brackets/quotes)")
        print(f"  Chars removed — HTML strip:          {stats['html_removed_chars']:,}")
        print(f"  Chars removed — URL strip:           {stats['url_removed_chars']:,}")
        print(f"  UTF-8 bytes of saved `text` only:    {_fmt_bytes(stats['cleaned_text_bytes_written'])}")
        if tr > 0 and stats["cleaned_text_bytes_written"] > 0:
            print(f"  Raw text / clean text (bytes):       {tr / stats['cleaned_text_bytes_written']:.2f}×")
        print(f"  Written: {BLOAT_REPORT_PATH}")
        print("  (Stripping HTML/URLs usually helps LM training: less noise, plain-text target.)")
        with open(LOG_FILE, "a", encoding="utf-8") as flog:
            flog.write("\n" + json.dumps(report, indent=2) + "\n")

    log_step(
        "3-clean",
        f"finished scanned={stats['total']:,} saved={stats['saved']:,} dupes={stats['dupes']:,} "
        f"short={stats['short']:,} errors={stats['errors']:,}",
    )
    return stats


process_all_datasets()


Started: 2026-03-29T13:10:41
Processing: zolai_corpus_master.jsonl
  scanned 250,000 | saved 249,972
  scanned 500,000 | saved 499,972
  scanned 750,000 | saved 749,972
  scanned 1,000,000 | saved 999,972
  scanned 1,250,000 | saved 1,249,972
  scanned 1,500,000 | saved 1,499,972
  scanned 1,750,000 | saved 1,749,972
  scanned 2,000,000 | saved 1,970,805


In [ ]:
# =========================
# 4. PEEK SAMPLES
# =========================
import json
import random


def peek_at_dataset(file_path, sample_size=10):
    file_path = Path(file_path)
    if not file_path.is_file():
        log_step("4-peek", f"skip missing {file_path}")
        print("Missing:", file_path)
        return
    log_step("4-peek", f"sampling n={sample_size} from {file_path.name}")
    samples = []
    with open(file_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i < sample_size:
                samples.append(line)
            else:
                j = random.randint(0, i)
                if j < sample_size:
                    samples[j] = line
    print("Samples from", file_path.name)
    for i, s in enumerate(samples):
        d = json.loads(s)
        print(f"[{i+1}]", (d.get("text") or "")[:500])
        if d.get("translation_en"):
            print("    EN:", str(d["translation_en"])[:200])
    log_step("4-peek", "done")


peek_at_dataset(PRIMARY_JSONL)


In [ ]:
# =========================
# 5. VERIFY + LENGTH HISTOGRAM
# =========================
import random
from collections import Counter

import matplotlib.pyplot as plt


def verify_and_stats(file_path, vocab_sample_lines=500_000, sample_size=12):
    file_path = Path(file_path)
    if not file_path.is_file():
        log_step("5-verify", f"skip missing {file_path}")
        print("Missing:", file_path)
        return
    log_step("5-verify", f"histogram + vocab sample (first {vocab_sample_lines:,} lines for vocab)")
    lengths = []
    vocab_words = []
    samples = []
    with open(file_path, "r", encoding="utf-8") as f:
        _it = enumerate(f)
        if USE_TQDM:
            from tqdm.auto import tqdm

            _it = tqdm(_it, desc="verify", unit="line", mininterval=2.0)
        for i, line in _it:
            d = json.loads(line)
            text = d.get("text") or ""
            w = text.split()
            lengths.append(len(w))
            if i < vocab_sample_lines:
                vocab_words.extend(w)
            if len(samples) < sample_size:
                samples.append(text)
            else:
                j = random.randint(0, i)
                if j < sample_size:
                    samples[j] = text
    print("Rows:", f"{len(lengths):,}", "| avg words:", f"{sum(lengths)/len(lengths):.1f}")
    cw = Counter(x.lower() for x in vocab_words)
    print("Top 15 tokens:")
    for w, c in cw.most_common(15):
        print(f"  {w[:40]!r}: {c:,}")
    hi = max(lengths) + 1 if lengths else 1
    plt.figure(figsize=(10, 4))
    plt.hist(lengths, bins=50, range=(0, min(80, hi)), color="steelblue", edgecolor="black")
    plt.title("Words per line")
    plt.xlabel("Word count")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
    log_step("5-verify", f"done rows={len(lengths):,}")


verify_and_stats(PRIMARY_JSONL)


In [ ]:
# =========================
# 6. AGGREGATE STATS (streaming)
# =========================
import json
from collections import Counter


def get_dataset_stats(file_path, max_lines=None):
    file_path = Path(file_path)
    if not file_path.is_file():
        log_step("6-stats", f"skip missing {file_path}")
        print("Missing:", file_path)
        return
    log_step("6-stats", f"streaming stats max_lines={max_lines}")
    n = 0
    words_total = 0
    lengths = []
    vocab = Counter()
    with open(file_path, "r", encoding="utf-8") as f:
        _it = f
        if USE_TQDM:
            from tqdm.auto import tqdm

            _it = tqdm(f, desc="stats", unit="line", mininterval=2.0)
        for line in _it:
            if max_lines is not None and n >= max_lines:
                break
            n += 1
            d = json.loads(line)
            ws = (d.get("text") or "").split()
            lengths.append(len(ws))
            words_total += len(ws)
            for w in ws:
                vocab[w.lower()] += 1
    avg = sum(lengths) / max(len(lengths), 1)
    print(
        f"{file_path.name}: lines={n:,} | words={words_total:,} | unique≈{len(vocab):,} | avg len={avg:.2f}"
    )
    log_step("6-stats", f"done lines={n:,} | words={words_total:,} | vocab≈{len(vocab):,}")


get_dataset_stats(PRIMARY_JSONL)


## Optional: soft refine

Strip common **English UI** substrings. Writes `SOFT_JSONL`. Toggle `RUN_SOFT_REFINE` in config.


In [ ]:
# =========================
# 7. SOFT REFINE
# =========================
import json
import re

NOISE_PATTERNS = [
    r"(?i)read more\s*[:-]*",
    r"(?i)\bcookie(s)?\b",
    r"(?i)fake account",
    r"(?i)sign up",
    r"(?i)log in",
    r"(?i)click here",
    r"(?i)page \d+\s+of\s+\d+",
    r"(?i)copyright\s+©?",
    r"(?i)subscribe\s+now",
]
_COMPILED = [re.compile(p) for p in NOISE_PATTERNS]


def run_soft_refine():
    if not RUN_SOFT_REFINE:
        log_step("7-soft", "skipped RUN_SOFT_REFINE=False")
        print("RUN_SOFT_REFINE is False — skip.")
        return
    if not PRIMARY_JSONL.is_file():
        log_step("7-soft", f"missing {PRIMARY_JSONL}")
        print("Run process cell first:", PRIMARY_JSONL)
        return
    log_step("7-soft", "start soft refine")
    n = ch = 0
    with open(PRIMARY_JSONL, "r", encoding="utf-8") as fin, open(SOFT_JSONL, "w", encoding="utf-8") as fout:
        _it = fin
        if USE_TQDM:
            from tqdm.auto import tqdm

            _it = tqdm(fin, desc="soft refine", unit="line", mininterval=2.0)
        for line in _it:
            n += 1
            d = json.loads(line)
            t0 = d.get("text") or ""
            t = t0
            for c in _COMPILED:
                t = c.sub(" ", t)
            t = re.sub(r"\s+", " ", t).strip()
            if t != t0:
                ch += 1
            d["text"] = t
            fout.write(json.dumps(d, ensure_ascii=False) + "\n")
            if (not USE_TQDM) and n % 200_000 == 0:
                print(f"  {n:,} lines, {ch:,} touched")
    print("Soft refine done:", SOFT_JSONL, "| lines", n, "| changed", ch)
    log_step("7-soft", f"done lines={n:,} changed={ch:,} -> {SOFT_JSONL.name}")


run_soft_refine()


## Optional: drop noisy rows

Drops lines whose lowercase text contains **functional English** phrases. Enable with `RUN_NOISE_ROW_DROP = True`. Writes `GOLD_JSONL`.


In [ ]:
# =========================
# 8. DROP FUNCTIONAL-NOISE ROWS
# =========================
import json

FUNCTIONAL_PHRASES = (
    "sign in",
    "log in",
    "sign up",
    "create account",
    "forgot password",
    "click here",
    "privacy policy",
    "terms of use",
    "all rights reserved",
    "powered by",
)


def run_noise_drop():
    if not RUN_NOISE_ROW_DROP:
        log_step("8-noise", "skipped RUN_NOISE_ROW_DROP=False")
        print("RUN_NOISE_ROW_DROP is False — skip (split uses soft or primary).")
        return
    src = SOFT_JSONL if SOFT_JSONL.is_file() else PRIMARY_JSONL
    if not src.is_file():
        log_step("8-noise", f"no source {src}")
        print("No source:", src)
        return
    log_step("8-noise", f"start from {src.name}")
    kept = drop = 0
    with open(src, "r", encoding="utf-8") as fin, open(GOLD_JSONL, "w", encoding="utf-8") as fout:
        _it = fin
        if USE_TQDM:
            from tqdm.auto import tqdm

            _it = tqdm(fin, desc="noise drop", unit="line", mininterval=2.0)
        for line in _it:
            d = json.loads(line)
            t = (d.get("text") or "").lower()
            if any(p in t for p in FUNCTIONAL_PHRASES):
                drop += 1
                continue
            fout.write(json.dumps(d, ensure_ascii=False) + "\n")
            kept += 1
    print("Noise drop: kept", kept, "dropped", drop, "->", GOLD_JSONL)
    log_step("8-noise", f"done kept={kept:,} dropped={drop:,}")


run_noise_drop()


In [ ]:
# =========================
# 9. RESOLVE SPLIT SOURCE
# =========================


def resolve_split_source() -> Path:
    if RUN_NOISE_ROW_DROP and GOLD_JSONL.is_file():
        return GOLD_JSONL
    if RUN_SOFT_REFINE and SOFT_JSONL.is_file():
        return SOFT_JSONL
    return PRIMARY_JSONL


SPLIT_SOURCE = resolve_split_source()
log_step("9-split-src", f"using {SPLIT_SOURCE}")
print("Split source:", SPLIT_SOURCE)


In [ ]:
# =========================
# 10. TRAIN / VAL / TEST SPLIT (98 / 1 / 1)
# =========================
import gc

import pandas as pd
from sklearn.model_selection import train_test_split


def create_splits(input_file: Path, out_dir: Path = None):
    out_dir = out_dir or OUTPUT_DIR
    input_file = Path(input_file)
    if not input_file.is_file():
        log_step("10-split", f"missing {input_file}")
        print("Missing:", input_file)
        return
    log_step("10-split", f"start load_json {input_file.name} (RAM heavy)")
    print("Loading (needs RAM for full dataframe)...", input_file.name)
    df = pd.read_json(input_file, lines=True)
    if "text" not in df.columns:
        raise KeyError("Expected column `text`")
    df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    train, temp = train_test_split(df, test_size=VAL_TEST_FRAC, random_state=SEED)
    val, test = train_test_split(temp, test_size=TEST_WITHIN_TEMP_FRAC, random_state=SEED)
    train.to_json(out_dir / "train.jsonl", orient="records", lines=True, force_ascii=False)
    val.to_json(out_dir / "val.jsonl", orient="records", lines=True, force_ascii=False)
    test.to_json(out_dir / "test.jsonl", orient="records", lines=True, force_ascii=False)
    del df, train, temp, val, test
    gc.collect()
    print("Wrote train/val/test to", out_dir)
    log_step("10-split", f"done wrote train/val/test.jsonl under {out_dir}")


create_splits(SPLIT_SOURCE)


In [ ]:
# =========================
# 11. HEALTH AUDIT (sample)
# =========================
import gc
import json
import random
from collections import Counter


def audit_zolai_sample(file_path: Path, max_rows: int = 500_000):
    if not file_path.is_file():
        log_step("11-audit", f"missing {file_path}")
        print("Missing:", file_path)
        return
    log_step("11-audit", f"sample up to {max_rows:,} rows from {file_path.name}")
    texts = []
    with open(file_path, "r", encoding="utf-8") as f:
        _it = enumerate(f)
        if USE_TQDM:
            from tqdm.auto import tqdm

            _it = tqdm(_it, desc="audit", unit="line", mininterval=2.0)
        for i, line in _it:
            if i >= max_rows:
                break
            texts.append(json.loads(line).get("text") or "")
    chunk = "".join(texts[:50_000])
    print("Glottal / apostrophe count in sample:", chunk.count("'"))
    words = " ".join(texts).lower().split()
    vocab = Counter(words)
    print("Unique words (sample):", f"{len(vocab):,}")
    print("Avg len:", f"{sum(len(t.split()) for t in texts) / len(texts):.1f}")
    for s in random.sample(texts, min(5, len(texts))):
        print(" ", s[:120], "...")
    del texts, vocab
    gc.collect()
    log_step("11-audit", "done")


audit_zolai_sample(SPLIT_SOURCE)


## Optional: Hugging Face `save_to_disk`

Set `EXPORT_HF_DATASET = True` in the config cell. Point **`zolai-qwen-training-v2.ipynb`** at `HF_EXPORT_DIR` via **`ZOLOAI_DATASET_SRC`**.


In [ ]:
# =========================
# 12. EXPORT DatasetDict → disk
# =========================
from datasets import Dataset, DatasetDict


def export_hf_disk():
    if not EXPORT_HF_DATASET:
        log_step("12-hf-export", "skipped EXPORT_HF_DATASET=False")
        print("EXPORT_HF_DATASET is False — skip.")
        return
    tr = OUTPUT_DIR / "train.jsonl"
    va = OUTPUT_DIR / "val.jsonl"
    if not tr.is_file() or not va.is_file():
        log_step("12-hf-export", "missing train/val jsonl")
        print("Need train.jsonl and val.jsonl in", OUTPUT_DIR)
        return
    log_step("12-hf-export", "loading JSON into DatasetDict (may take a while)")
    d = DatasetDict(
        {
            "train": Dataset.from_json(str(tr)),
            "validation": Dataset.from_json(str(va)),
        }
    )
    te = OUTPUT_DIR / "test.jsonl"
    if te.is_file():
        d["test"] = Dataset.from_json(str(te))
    HF_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    log_step("12-hf-export", f"save_to_disk -> {HF_EXPORT_DIR}")
    d.save_to_disk(str(HF_EXPORT_DIR))
    print("Saved:", HF_EXPORT_DIR)
    print("Set ZOLOAI_DATASET_SRC=", HF_EXPORT_DIR)
    log_step("12-hf-export", "done")


export_hf_disk()


## Kaggle dataset: new version

Stage files + `dataset-metadata.json`, then run `kaggle datasets version`. Uncomment `subprocess.run` when ready.


In [ ]:
# =========================
# 13. KAGGLE DATASET VERSION
# =========================
import json
import shutil
import subprocess

log_step("13-kaggle", "staging files for dataset version")
UPLOAD_STAGING = WORK_DIR / "zolai_kaggle_upload_staging"
UPLOAD_STAGING.mkdir(parents=True, exist_ok=True)

FILES_TO_PUBLISH = [
    OUTPUT_DIR / "zolai_train_ready_v2.jsonl",
    OUTPUT_DIR / "train.jsonl",
    OUTPUT_DIR / "val.jsonl",
    OUTPUT_DIR / "test.jsonl",
]
if LOG_SHORT_TEXT:
    FILES_TO_PUBLISH.append(SHORT_LOG_PATH)

metadata = {
    "id": KAGGLE_DATASET_ID,
    "title": "Zolai master / cleaned corpus",
    "licenses": [{"name": "CC0-1.0"}],
}

for p in FILES_TO_PUBLISH:
    if Path(p).is_file():
        shutil.copy2(p, UPLOAD_STAGING / Path(p).name)
        print("Staged:", p.name)
    else:
        print("Skip (missing):", p)

with open(UPLOAD_STAGING / "dataset-metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

cmd = ["kaggle", "datasets", "version", "-p", str(UPLOAD_STAGING), "-m", KAGGLE_VERSION_MESSAGE]
print("Command:", " ".join(cmd))
# subprocess.run(cmd, check=False)
print("Uncomment subprocess.run to publish.")
log_step("13-kaggle", "staging ready (subprocess.run commented)")


## Optional: chunk large JSONL (~100 MB parts)


In [ ]:
# =========================
# 14. CHUNK JSONL BY SIZE
# =========================


def chunk_jsonl(input_file, output_dir, chunk_mb: int = 100):
    input_file = Path(input_file)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    log_step("14-chunk", f"start {input_file.name} -> {output_dir} ({chunk_mb} MiB parts)")
    limit = chunk_mb * 1024 * 1024
    part = 0
    size = 0
    out = open(output_dir / f"part_{part}.jsonl", "w", encoding="utf-8")
    with open(input_file, "r", encoding="utf-8") as f:
        _it = f
        if USE_TQDM:
            from tqdm.auto import tqdm

            _it = tqdm(f, desc="chunk", unit="line", mininterval=2.0)
        for line in _it:
            b = len(line.encode("utf-8"))
            if size + b > limit and size > 0:
                out.close()
                part += 1
                size = 0
                out = open(output_dir / f"part_{part}.jsonl", "w", encoding="utf-8")
            out.write(line)
            size += b
    out.close()
    print("Parts:", part + 1, "in", output_dir)
    log_step("14-chunk", f"done parts={part + 1}")


# chunk_jsonl(PRIMARY_JSONL, WORK_DIR / "zolai_jsonl_chunks", chunk_mb=100)
print("Uncomment chunk_jsonl(...) to run.")


## Appendix: custom BPE tokenizer

Only for **from-scratch** base LMs. **Qwen LoRA** uses `zolai-qwen-training-v2.ipynb` + Hub tokenizer — skip this.


In [ ]:
# =========================
# APPENDIX — optional BPE
# =========================
import json

RUN_CUSTOM_TOKENIZER = False


def train_custom_bpe_optional():
    if not RUN_CUSTOM_TOKENIZER:
        log_step("bpe", "skipped RUN_CUSTOM_TOKENIZER=False")
        print("RUN_CUSTOM_TOKENIZER is False — skip.")
        return
    from tokenizers import Tokenizer, models, pre_tokenizers, trainers

    src = GOLD_JSONL if GOLD_JSONL.is_file() else PRIMARY_JSONL
    if not src.is_file():
        log_step("bpe", f"missing {src}")
        print("No jsonl:", src)
        return
    log_step("bpe", f"training tokenizer from {src.name}")
    tok = Tokenizer(models.BPE(unk_token="[UNK]"))
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    trainer = trainers.BpeTrainer(
        vocab_size=32000,
        min_frequency=2,
        special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]", "<s>", "</s>"],
    )

    def it():
        with open(src, "r", encoding="utf-8") as f:
            for line in f:
                yield json.loads(line)["text"]

    tok.train_from_iterator(it(), trainer=trainer)
    out = WORK_DIR / "zolai_tokenizer.json"
    tok.save(str(out))
    print("Saved:", out)
    log_step("bpe", f"saved {out}")


train_custom_bpe_optional()


## Training with Qwen

Use **`zolai-qwen-training-v2.ipynb`** after exporting data (JSONL splits or `save_to_disk`). From-scratch `Trainer` blocks were removed from this notebook to avoid conflicting with the LoRA pipeline.
